# 🚀 Entrenamiento en la nube — Colab / Kaggle

Este notebook **no** es educativo (para eso están 00–08): es el *runner* que entrena **tu** modelo con el pipeline de `scripts/`.

**Flujo recomendado**

| Dónde | Qué | Costo |
|---|---|---|
| Mac, contenedor `LLMLab` | `make data PROFILE=small WORKERS=10` → `make bundle PROFILE=small` | gratis |
| Drive / Kaggle Dataset | subir `dist/llm-lab-small.tar` (~0.7 GB, **privado**) | gratis |
| Este notebook (T4) | pre-entrenar ~17 M parámetros sobre ~350 M tokens (estimado ~1.5–2 h, no medido) + SFT | gratis |

El entrenamiento **guarda `last.pt` cada 250 pasos** en Drive (Colab) o `/kaggle/working` (Kaggle). Si la sesión se corta, vuelve a ejecutar todas las celdas: `pretrain.py` reanuda solo, con optimizador, scheduler, RNG y DataLoader exactamente donde quedaron.

> ⚠️ El `.tar` contiene tokens de tu corpus personal: mantenlo privado (Drive personal o Kaggle Dataset *private*).

## 1 · Entorno

In [ ]:
import os, subprocess, sys
PLATFORM = "kaggle" if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") else "colab"
PROFILE = "small"            # "small" en T4 · "large" en A100/L4 (Colab Pro)
print("Plataforma:", PLATFORM)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
if PLATFORM == "colab":
    from google.colab import drive
    drive.mount("/content/drive")
    BUNDLE  = f"/content/drive/MyDrive/llm-lab/llm-lab-{PROFILE}.tar"
    RUN_DIR = f"/content/drive/MyDrive/llm-lab/runs/{PROFILE}"   # checkpoints persistentes
    WORK    = "/content/llm"
else:
    # Kaggle: sube el .tar como Dataset privado llamado "llm-lab-data" y agrégalo como input.
    BUNDLE  = f"/kaggle/input/llm-lab-data/llm-lab-{PROFILE}.tar"
    RUN_DIR = f"/kaggle/working/runs/{PROFILE}"                   # se conserva al "Save Version"
    WORK    = "/tmp/llm"                                         # disco local de la VM
os.makedirs(RUN_DIR, exist_ok=True)
print(BUNDLE, "→", WORK, "| checkpoints en", RUN_DIR)

## 2 · Código + datos

Se extrae el bundle al disco **local** de la VM: leer los `.bin` por memmap desde Drive (FUSE) sería 10–50× más lento.

In [ ]:
os.makedirs(WORK, exist_ok=True)
!tar -xf "{BUNDLE}" -C "{WORK}"
%cd {WORK}
!pip install -q -r requirements.txt && pip install -q --no-deps -e .
!ls -lh datasets/tokenized/{PROFILE}

In [ ]:
import json
meta = json.load(open(f"datasets/tokenized/{PROFILE}/meta.json"))
print(f"vocab {meta['vocab_size']:,} · {meta['effective_train_tokens']/1e6:.0f} M tokens efectivos")
for src, s in meta["sources"].items():
    print(f"  {src:15s} {s['train_tokens']/1e6:8.2f} M × {s['repeat']}")
print("dominios:", {k: f"{v:.1%}" for k, v in list(meta["domains"].items())[:10]})

## 3 · Pre-entrenamiento

Reanuda automáticamente si `RUN_DIR/last.pt` existe. Para probar rápido antes del run largo: agrega `--max_steps 50`.

*Colab Free*: deja la pestaña abierta; las sesiones duran unas horas. *Kaggle*: usa **Save Version → Save & Run All** para que corra en segundo plano (hasta 12 h) sin depender del navegador.

In [ ]:
!python scripts/pretrain.py --profile {PROFILE} --out_dir "{RUN_DIR}"

## 4 · Curvas de entrenamiento

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
log = pd.read_json(f"{RUN_DIR}/logs/metrics_{PROFILE}.jsonl", lines=True)
tr, ev = log.dropna(subset=["loss"]), log.dropna(subset=["val_loss"])
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
ax[0].plot(tr.step, tr.loss, alpha=.4, label="train"); ax[0].plot(ev.step, ev.val_loss, "o-", label="val")
if "val_personal_loss" in ev: ax[0].plot(ev.step, ev.val_personal_loss, "s-", label="val personal")
ax[0].set(title="Cross-entropy (nats/token)", xlabel="step"); ax[0].legend()
ax[1].plot(tr.step, tr.lr); ax[1].set(title="Learning rate (warmup + coseno)", xlabel="step")
ax[2].plot(tr.step, tr.tokens_per_sec); ax[2].set(title="Throughput (tokens/s)", xlabel="step")
plt.tight_layout(); plt.show()
print(ev[[c for c in ev.columns if c.startswith("val")]].tail(3))

## 5 · Generación

In [ ]:
for p in ["La inferencia bayesiana", "El algoritmo genético", "Según Kant, la ética",
          "La armonía tonal", "def train_model("]:
    !python scripts/sample.py --run "{RUN_DIR}" --prompt "{p}" --max_new_tokens 80
    print("-" * 80)

## 6 · SFT (formato de asistente)

Fine-tuning instruccional sobre `datasets/sft`. Con el set semilla aprende el **formato** pregunta→respuesta; el conocimiento viene del pre-entrenamiento.

In [ ]:
!python scripts/sft.py --run "{RUN_DIR}" --epochs 4
!python scripts/sample.py --run "{RUN_DIR}-sft" --chat "¿Qué diferencia hay entre RAG y fine-tuning?" --temperature 0.5

## 7 · Guardar resultados

- **Colab**: `RUN_DIR` ya está en Drive.
- **Kaggle**: al terminar el *Save Version*, la carpeta `/kaggle/working/runs` queda como output; para reanudar en otra sesión, agrega ese output como input y cópialo a `RUN_DIR` antes del paso 3:
  `!cp -r /kaggle/input/<tu-notebook>/runs/{PROFILE}/* {RUN_DIR}/`